# Lab 4.3 — The double-slit experiment, one electron at a time

**Goal:** compute an electron's de Broglie wavelength, predict its interference pattern, then simulate thousands of electrons landing one by one and watch the fringes appear.

An electron accelerated through a voltage $V$ has momentum $p = \sqrt{2 m e V}$ and wavelength

$$\lambda = \frac{h}{p} = \frac{h}{\sqrt{2 m e V}}$$

Behind two slits a distance $d$ apart, each of width $a$, on a screen a distance $L$ away, the chance of landing at height $y$ is

$$I(y) \propto \cos^2\!\left(\frac{\pi d y}{\lambda L}\right)\,\mathrm{sinc}^2\!\left(\frac{\pi a y}{\lambda L}\right)$$

with bright fringes $\Delta y = \lambda L / d$ apart.

In [ ]:
import sys; sys.path.append("..")
from constants import h, m_e, e
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(seed=4)

L = 1.0            # slits to screen, m
a = 0.2e-6         # slit width, m
d = 1.0e-6         # slit separation, m
Y = 60e-6          # screen shows -Y .. +Y

## 1. De Broglie wavelength

Complete `de_broglie(V)` and check it for a few voltages.

In [ ]:
def de_broglie(V):
    # Wavelength in metres of an electron accelerated through V volts
    return h / np.sqrt(2 * m_e * e * V)

for V in [100, 1_000, 10_000, 50_000]:
    lam = de_broglie(V)
    print(f"{V:>6} V:  lambda = {lam*1e12:7.2f} pm   fringe spacing = {lam*L/d*1e6:6.2f} um")

## 2. The predicted pattern

Three setups: both slits open, a detector watching which slit, and one slit closed.

In [ ]:
def sinc2(x):
    return np.sinc(x / np.pi) ** 2      # numpy's sinc(x) is sin(pi x)/(pi x)

def single(y, yc, lam):
    return sinc2(np.pi * a * (y - yc) / (lam * L))

def intensity(y, lam, mode="both"):
    if mode == "one":
        return single(y, d / 2, lam)
    if mode == "watch":
        return 0.5 * (single(y, d / 2, lam) + single(y, -d / 2, lam))
    return np.cos(np.pi * d * y / (lam * L)) ** 2 * single(y, 0, lam)

lam = de_broglie(10_000)
y = np.linspace(-Y, Y, 2000)
plt.figure(figsize=(8, 4))
for mode, label in [("both", "two slits"), ("watch", "two slits, detector on"), ("one", "one slit")]:
    plt.plot(y * 1e6, intensity(y, lam, mode), label=label)
plt.xlabel("Position on screen (um)")
plt.ylabel("Relative probability")
plt.title("Electron double slit at 10 kV")
plt.legend()
plt.show()

## 3. Fire electrons one at a time

Each electron lands at one random spot. We choose that spot by **rejection sampling**: pick a random $y$, keep it with probability $I(y)$, otherwise try again. Since $I(y) \le 1$, kept points follow the pattern exactly.

In [ ]:
def fire(n, lam, mode="both"):
    hits = []
    while len(hits) < n:
        ys = rng.uniform(-Y, Y, size=4 * n)
        keep = rng.uniform(0, 1, size=ys.size) < intensity(ys, lam, mode)
        hits.extend(ys[keep])
    return np.array(hits[:n])

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
for ax, n in zip(axes, [10, 100, 1_000, 20_000]):
    hits = fire(n, lam)
    ax.hist(hits * 1e6, bins=80, range=(-Y * 1e6, Y * 1e6))
    ax.set_title(f"{n:,} electrons")
    ax.set_xlabel("y (um)")
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. Does the simulation match the theory?

In [ ]:
N = 50_000
for mode in ["both", "watch"]:
    hits = fire(N, lam, mode)
    counts, edges = np.histogram(hits, bins=80, range=(-Y, Y))
    centres = (edges[:-1] + edges[1:]) / 2
    expected = intensity(centres, lam, mode)
    expected *= N / expected.sum()
    plt.figure(figsize=(8, 3))
    plt.bar(centres * 1e6, counts, width=(edges[1] - edges[0]) * 1e6, alpha=0.6, label="simulated hits")
    plt.plot(centres * 1e6, expected, color="black", label="theory")
    plt.title(f"mode = {mode}")
    plt.xlabel("y (um)")
    plt.legend()
    plt.show()
    print(f"{mode}: largest difference from theory = {np.max(np.abs(counts - expected)) / np.sqrt(expected.max()):.1f} standard deviations")

## Newton vs quantum

A Newtonian particle goes through one slit on a straight line, so two open slits could only ever *add* hits. Find a spot where two slits give **fewer** electrons than one slit does:

In [ ]:
one = intensity(y, lam, "one")
both = intensity(y, lam, "both") * 2   # two slits let through twice as many electrons in total
darker = y[(both < 0.5 * one) & (one > 0.2)]
print(f"At y = {darker[0]*1e6:.2f} um, opening the second slit cuts the hit rate by more than half.")

## Exercises

1. **[Core]** What accelerating voltage gives fringes 20 um apart with $d = 1$ um?
2. **[Extension]** Double the slit width $a$. What happens to the envelope, and to the number of fringes you can see?
3. **[Challenge]** A 57 g tennis ball moves at 20 m/s through two gaps 10 cm apart, 5 m from a wall. Compute its fringe spacing. Could anything ever measure it?